# Шаг 03 — ответ как Python-объект

Когда ответ читает код, а не человек, текст неудобен. Описываем форму ответа Pydantic-моделью — `with_structured_output` вернёт проверенный объект.

In [ ]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()  # reads OPENROUTER_API_KEY and MODEL from .env

llm = ChatOpenAI(
    model=os.getenv("MODEL", "openai/gpt-4o-mini"),
    base_url="https://openrouter.ai/api/v1",   # OpenRouter speaks the OpenAI protocol
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class Task(BaseModel):
    """A development task extracted from a free-form request."""

    title: str = Field(description="Short task title")
    priority: Literal["low", "medium", "high"]
    steps: list[str] = Field(description="2-4 concrete steps to do it")

`method="function_calling"` передаёт схему как инструмент — работает на любой модели с tool calling.

In [ ]:
extractor = llm.with_structured_output(Task, method="function_calling")

task = extractor.invoke(
    "Срочно! Падает парсер логов на пустых строках, надо починить и добавить тесты."
)
task

In [ ]:
print(type(task).__name__, "|", task.priority)
for i, step in enumerate(task.steps, 1):
    print(f"{i}. {step}")

## Три способа получить структуру

У `with_structured_output` есть параметр `method` — **как именно сообщить модели форму ответа**:

| method | что уходит в API | что гарантирует |
|---|---|---|
| `function_calling` | схема как **инструмент** (`tools` + `tool_choice` — «обязан вызвать») | работает на любой модели с tool calling |
| `json_mode` | только `response_format: {"type": "json_object"}` — **схема не отправляется** | лишь то, что ответ — валидный JSON |
| `json_schema` *(по умолчанию)* | `response_format` с полной JSON-схемой | провайдер ограничивает генерацию строго под схему |

`include_raw=True` возвращает сырой ответ, разобранный объект и ошибку разбора — и **не падает**, если ответ не подошёл под схему. Удобно, чтобы сравнить методы.

In [ ]:
PROMPT = "Срочно! Падает парсер логов на пустых строках, надо починить и добавить тесты."


def try_method(method: str, prompt: str = PROMPT) -> None:
    """Run one structured-output method and show raw reply, result and error."""
    extractor = llm.with_structured_output(Task, method=method, include_raw=True)
    result = extractor.invoke(prompt)
    raw = result["raw"]
    print("raw:   ", raw.tool_calls[0]["args"] if raw.tool_calls else raw.content)
    print("parsed:", result["parsed"])
    print("error: ", result["parsing_error"])

### `function_calling` — схема как инструмент

Модель «вызывает функцию» `Task`, ответ приходит в `tool_calls`, LangChain превращает аргументы в объект.

In [ ]:
try_method("function_calling")

### `json_mode` — «просто ответь JSON-ом»

Схема **не отправляется**: модель знает только, что нужен JSON, но не знает ключей. Скорее всего она придумает свой формат — и Pydantic его отвергнет.

In [ ]:
try_method("json_mode")

Чтобы `json_mode` заработал, форму ответа нужно **описать в промпте самим**:

In [ ]:
try_method("json_mode", PROMPT + " Ответь JSON с ключами: title (строка), "
           "priority (low | medium | high), steps (список строк).")

### `json_schema` — нативные Structured Outputs

Провайдер получает схему и ограничивает генерацию: ответ физически не может от неё отличаться. Самый надёжный режим — но модель и провайдер должны его поддерживать.

In [ ]:
try_method("json_schema")

**Итог.** `json_schema` — самый строгий, если модель его поддерживает; `function_calling` — самый переносимый (поэтому он у нас выше); `json_mode` — только с описанием формата в промпте.

Заметь: язык ответа тоже «наследуется» из схемы — описания полей на английском могут увести ответ в английский. Если нужен русский, напиши это в описании полей или в системном промпте.